In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import json
import time
import random
import requests
import gc
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from datetime import datetime

# ============================================================
# 1. KAGGLE GPU & ENVIRONMENT SETUP
# ============================================================
def seed_everything(seed=42):
    """Ensures deterministic behavior and reproducibility across runs."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

print("Checking hardware acceleration...")
if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"✅ GPU Available! Number of GPUs found: {device_count}")
    for i in range(device_count):
        print(f"   👉 GPU {i}: {torch.cuda.get_device_name(i)}")
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️ No GPU detected. Running on CPU instead.")

NUM_WORKERS = 2 if torch.cuda.is_available() else 0
print(f"DataLoader workers optimized to: {NUM_WORKERS}\n")

# ============================================================
# 2. CONFIGURATIONS & HYPERPARAMETERS
# ============================================================
# Pointing paths directly to your explicit subfolders on disk
TRAIN_DATASET_PATH = '/kaggle/input/datasets/dhruvtaneja31/m-rice-dataset/processed_dataset_v3/train'
TEST_DATASET_PATH  = '/kaggle/input/datasets/dhruvtaneja31/m-rice-dataset/processed_dataset_v3/test'

NUM_CLASSES  = 4
BATCH_SIZE   = 32
MAX_EPOCHS   = 30  # Optimized training window
PATIENCE     = 7   # Allows the scheduler room to stabilize learning
IMG_SIZE     = 224
N_FOLDS      = 5
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working/'
SHEET_URL    = "https://script.google.com/macros/s/AKfycbyjpRwDy8bUdOGobfckfVuHy-32GNCxkmX1q-yxRiKKZqzsVmXn-123sYgMzVSw1ev1og/exec"

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# 3. DATA AUGMENTATION PIPELINES
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ============================================================
# 4. MODEL BUILDER FACTORY
# ============================================================
def get_efficientnet(variant, strategy, num_classes=4, dropout=0.5):
    model_fn = {
        'b0': models.efficientnet_b0, 'b1': models.efficientnet_b1,
        'b2': models.efficientnet_b2, 'b3': models.efficientnet_b3,
        'b4': models.efficientnet_b4,
    }[variant]

    model = model_fn(weights='DEFAULT')

    if strategy == 'transfer':
        for param in model.features.parameters(): param.requires_grad = False
    elif strategy == 'finetune_last2':
        for param in model.features.parameters(): param.requires_grad = False
        for param in model.features[-2:].parameters(): param.requires_grad = True
    elif strategy == 'full':
        for param in model.parameters(): param.requires_grad = True

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes)
    )
    return model

# ============================================================
# 5. LOGGING UTILITY
# ============================================================
def log_to_sheet(exp_id, dataset, preprocessing, model, model_name,
                 train_accs, val_accs, train_losses, val_losses,
                 phase, lr, patience, max_epochs,
                 batch_size, img_size, fold, notes=''):

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params    = total_params - trainable_params
    total_layers     = len(list(model.modules()))
    trainable_layers = len([l for l in model.modules() if any(p.requires_grad for p in l.parameters(recurse=False))])
    frozen_layers    = total_layers - trainable_layers

    best_epoch    = int(np.argmax(val_accs)) + 1
    epochs_run    = len(val_accs)
    early_stopped = epochs_run < max_epochs

    row = [
        exp_id,
        datetime.now().strftime('%Y-%m-%d %H:%M'),
        dataset, preprocessing,
        model_name,
        f"{total_params:,}", f"{trainable_params:,}", f"{frozen_params:,}",
        frozen_layers, trainable_layers,
        img_size, batch_size, phase, str(lr), patience,
        epochs_run, max_epochs,
        f"{max(train_accs)*100:.2f}%",
        f"{max(val_accs)*100:.2f}%",
        f"{min(train_losses):.4f}",
        f"{min(val_losses):.4f}",
        best_epoch, str(early_stopped),
        f"Fold {fold}/{N_FOLDS} | {notes}"
    ]

    try:
        response = requests.post(SHEET_URL, data=json.dumps({"row": row}), headers={"Content-Type": "application/json"}, timeout=15)
        print(f"✅ Logged to sheet: {exp_id} | Fold {fold} | Best Cross-Val Acc: {max(val_accs)*100:.2f}%")
    except Exception as e:
        print(f"⚠️ Google Sheets logging failed: {e}")

# ============================================================
# 6. TRAINING ENGINE WITH ADVANCED CROSS-VALIDATION METRICS
# ============================================================
def train_fold(model, optimizer, scheduler, criterion, train_loader, val_loader,
               exp_id, fold, phase, lr, model_name, notes=''):

    train_accs, val_accs     = [], []
    train_losses, val_losses = [], []
    best_val_acc     = 0.0
    patience_counter = 0
    class_labels = list(range(NUM_CLASSES))

    for epoch in range(MAX_EPOCHS):
        # --- Train Phase ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
            
        train_accs.append(correct / total)
        train_losses.append(running_loss / len(train_loader))

        # --- Validation Phase (Per Fold) ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_targets, all_preds, all_probs = [], [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss    += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total   += labels.size(0)
                
                probs = F.softmax(outputs, dim=1)
                all_targets.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                
        val_accs.append(val_correct / val_total)
        val_losses.append(val_loss / len(val_loader))

        all_targets = np.array(all_targets, dtype=int)
        all_preds = np.array(all_preds, dtype=int)
        all_probs = np.array(all_probs, dtype=float)
        
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning)
            precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
        
        try:
            # ROBUST FIX: Evaluates metrics smoothly using One-vs-Rest configuration across multi-class predictions
            auc_score = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro', labels=class_labels)
        except Exception:
            auc_score = 0.5
            
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        print(f"  Epoch {epoch+1}/{MAX_EPOCHS} | LR: {current_lr:.6f}\n"
              f"  🏋️‍♂️  Train -> Acc: {train_accs[-1]*100:.2f}% | Loss: {train_losses[-1]:.4f}\n"
              f"  🧪 Val   -> Acc: {val_accs[-1]*100:.2f}% | Loss: {val_losses[-1]:.4f}\n"
              f"  📊 Scores-> Precision: {precision*100:.2f}% | Recall: {recall*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {auc_score*100:.2f}%")

        if val_accs[-1] > best_val_acc:
            best_val_acc     = val_accs[-1]
            patience_counter = 0
            save_path = os.path.join(SAVE_DIR, f'{exp_id}_fold{fold}_best.pth')
            torch.save({'model_state': model.state_dict()}, save_path)
            print(f"  💾 Saved Checkpoint: {exp_id}_fold{fold}_best.pth")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  ⏹ Early stopped at epoch {epoch+1}")
                break

    extended_notes = f"{notes} | Final F1: {f1*100:.1f}% | AUC: {auc_score*100:.1f}%"
    log_to_sheet(
        exp_id=exp_id, dataset='preprocessed_v3', preprocessing='Resize 224x224, Augmented, 5-Fold CV',
        model=model, model_name=model_name, train_accs=train_accs, val_accs=val_accs,
        train_losses=train_losses, val_losses=val_losses, phase=phase, lr=lr, patience=PATIENCE,
        max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, img_size=f'{IMG_SIZE}x{IMG_SIZE}', fold=fold, notes=extended_notes
    )
    return best_val_acc

# ============================================================
# 7. EVALUATION ENGINE FOR THE COLD-STORAGE UNSEEN TEST FOLDER
# ============================================================
def evaluate_test_set(model_path, test_loader, variant, strategy):
    """Loads the highest-performing fold checkpoint and runs evaluation against the genuine test directory."""
    print(f"\n🔒 RUNNING FINAL TEST EVALUATION AGAINST UNSEEN REPOSITORY...")
    model = get_efficientnet(variant, strategy).to(DEVICE)
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    test_correct, test_total = 0, 0
    all_targets, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            test_correct += predicted.eq(labels).sum().item()
            test_total   += labels.size(0)
            
            probs = F.softmax(outputs, dim=1)
            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_targets = np.array(all_targets, dtype=int)
    all_preds = np.array(all_preds, dtype=int)
    all_probs = np.array(all_probs, dtype=float)

    # Completely unmasked real-world score metrics (All 4 classes are guaranteed present here)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
    auc_score = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro', labels=list(range(NUM_CLASSES)))

    print(f"  🏁 ========================================== 🏁\n"
          f"  🏆 FINAL UNSEEN COLD-STORAGE TEST RESULTS 🏆\n"
          f"  🏁 ========================================== 🏁\n"
          f"  🎯 True Test Accuracy: {(test_correct/test_total)*100:.2f}%\n"
          f"  🔍 Test Precision: {precision*100:.2f}%\n"
          f"  🔍 Test Recall:    {recall*100:.2f}%\n"
          f"  📈 Test F1-Score:  {f1*100:.2f}%\n"
          f"  📈 Test ROC-AUC:   {auc_score*100:.2f}%\n"
          f"  🏁 ========================================== 🏁\n")
    return test_correct / test_total

# ============================================================
# 8. THE EXECUTION PIPELINE FOR FIXED CROSS VALIDATION
# ============================================================
base_train_dataset = ImageFolder(root=TRAIN_DATASET_PATH)
train_targets = [s[1] for s in base_train_dataset.samples]

# Instantiating separate datasets to keep evaluation inputs entirely clean and non-augmented
train_dataset_transformed = ImageFolder(root=TRAIN_DATASET_PATH, transform=train_transform)
val_dataset_transformed   = ImageFolder(root=TRAIN_DATASET_PATH, transform=val_transform)

# Pointing the dedicated validation loader directly to the static test fold path
test_dataset = ImageFolder(root=TEST_DATASET_PATH, transform=val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.CrossEntropyLoss()

variants   = ['b0', 'b1', 'b2', 'b3', 'b4']
strategies = ['transfer', 'finetune_last2', 'full']
exp_counter = 1
results     = {}

for variant in variants:
    for strategy in strategies:
        exp_id    = f'EFF_{variant.upper()}_{strategy}_EXP{exp_counter:03d}'
        model_name = f'EfficientNet-{variant.upper()}'
        fold_accs  = []
        
        # Lower baseline learning rate specifically for deep tuning to prevent gradient explosion
        current_lr = 0.0001 if strategy == 'full' else 0.001

        print(f"\n{'='*60}\n {exp_id} | {model_name} | Strategy: {strategy} | Base LR: {current_lr}\n{'='*60}")

        # The cross validation loop operates strictly over the 1,280 training indices pool
        for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_targets)), train_targets), 1):
            print(f"\n   Fold {fold}/{N_FOLDS}")

            train_subset = Subset(train_dataset_transformed, train_idx)
            val_subset   = Subset(val_dataset_transformed, val_idx)
            
            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
            val_loader   = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

            model     = get_efficientnet(variant, strategy).to(DEVICE)
            optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=current_lr)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

            best_acc = train_fold(
                model=model, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
                train_loader=train_loader, val_loader=val_loader,
                exp_id=exp_id, fold=fold, phase=strategy, lr=current_lr, model_name=model_name,
                notes=f'Adam+Cosine LR={current_lr} | {model_name} | {strategy}'
            )
            fold_accs.append(best_acc)
            
            del model, optimizer, scheduler, train_loader, val_loader
            torch.cuda.empty_cache()
            gc.collect()

        # Identify the best checkpoint among the 5 cross-validation folds
        best_fold_num = np.argmax(fold_accs) + 1
        winning_model_path = os.path.join(SAVE_DIR, f'{exp_id}_fold{best_fold_num}_best.pth')
        
        # Test the best fold checkpoint against the completely unseen test directory
        test_accuracy = evaluate_test_set(winning_model_path, test_loader, variant, strategy)
        results[exp_id] = test_accuracy 
        exp_counter += 1

# ============================================================
# 9. FINAL SUMMARY
# ============================================================
print(f"\n{'='*60}\nFINAL RESULTS SUMMARY (RANKED BY TRUE TEST ACCURACY)\n{'='*60}")
best_exp = max(results, key=results.get)
for exp_id, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{exp_id}: {acc*100:.2f}%")
print(f"\n 🏆 Best Architecture: {best_exp} | Unseen Test Folder Acc: {results[best_exp]*100:.2f}%")